# Day 2 · 1교시 · PTQ · AWQ 이론

> **VLM 경량화 2일 과정 · Day 2 (1교시) · 이론**
> 실습 환경: **RunPod A100 80GB** (이 교시는 이론 — 베이스 커널 + numpy만 사용) · 모델: **Qwen3-VL**

---

## 이 교시의 목표
- **PTQ vs QAT**의 차이와, PTQ가 *배포*에 적합한 이유를 설명한다.
- **AWQ 원리**(activation-aware: salient 채널 보존)를 numpy 데모로 체감한다.
- **GPTQ · FP8 · compressed-tensors**와의 관계를 정리한다.
- **VLM 양자화 범위**(LLM 선형층만, 비전 타워·프로젝터는 유지)와 **calibration**의 역할을 이해한다.


## 0. 공통 헤더 — RunPod 작업 폴더(VLM_FT_2) + HF_TOKEN 로드

> 📌 **모든 Day 2 노트북은 이 셀을 먼저 실행합니다.** RunPod 영구 볼륨의 작업 폴더 **`/workspace/VLM_FT_2`** 를 기준으로 잡고, `.env`의 **HF_TOKEN**을 불러옵니다. (Day2는 RunPod이라 Google Drive 마운트가 없습니다.)
> - `VLM_DIR` / `DATA_DIR` : 교시 간 공유 폴더(모델·AWQ 결과·평가/벤치 JSON이 여기 모입니다).
> - **HF_TOKEN**: `VLM_FT_2/.env` 에 `HF_TOKEN=hf_...` 한 줄을 넣어두면 자동 로드됩니다. `login()`은 호출하지 않습니다(환경변수만으로 충분).

In [1]:
# ════════════════════════════════════════════════════════════
#  공통 헤더 · RunPod 작업 폴더(VLM_FT_2) + HF_TOKEN(.env) 로드
#  (모든 Day2 노트북 상단에서 동일하게 실행)
# ════════════════════════════════════════════════════════════
import os

# (1) RunPod 영구 볼륨의 작업 폴더 VLM_FT_2 (교시 간 모델·결과 공유). Drive 마운트 불필요.
VLM_DIR = '/workspace/VLM_FT_2'
DATA_DIR = f'{VLM_DIR}/data'
os.makedirs(DATA_DIR, exist_ok=True)

# (2) .env 에서 HF_TOKEN 로드. login()은 부르지 않음(환경변수만으로 인증, 경고 없음).
try:
    from dotenv import load_dotenv
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'python-dotenv'])
    from dotenv import load_dotenv
ENV_PATH = f'{VLM_DIR}/.env'
if os.path.exists(ENV_PATH):
    load_dotenv(ENV_PATH)
    print('HF_TOKEN:', '로드됨' if os.environ.get('HF_TOKEN') else '없음')
else:
    print(f'.env 없음 — {ENV_PATH} 에 HF_TOKEN=hf_... 한 줄을 만들면 자동 로드됩니다(공개 모델만 쓰면 없어도 됨)')
print('작업 폴더 VLM_DIR =', VLM_DIR)

HF_TOKEN: 로드됨
작업 폴더 VLM_DIR = /workspace/VLM_FT_2


## 0. Day 1 → Day 2 — 무게중심이 바뀝니다

| | Day 1 (T4) | **Day 2 (A100 80GB)** |
|---|---|---|
| 목표 | 학습을 가볍게 | **배포를 가볍게** |
| 기법 | QLoRA(학습 중 4bit + 어댑터) | **AWQ**(학습 후 4bit) + **vLLM 서빙** |
| 양자화 시점 | 학습하면서 | **학습 끝난 모델에** |

Day 1의 결과물(병합 4B)과 베이스 8B를, 오늘 **배포용으로 4bit 양자화**하고 **서빙**해 속도·메모리 이득을 측정합니다.

## 1. PTQ vs QAT

| | **PTQ** (Post-Training Quantization) | **QAT** (Quantization-Aware Training) |
|---|---|---|
| 언제 | **학습이 끝난 뒤** | 학습 중에 양자화를 흉내내며 |
| 비용 | **싸다**(소수 calibration 데이터, 수 분~수십 분) | 비싸다(전체 재학습) |
| 정확도 | 좋음(4bit에서 실용적) | 약간 더 좋을 수 있음 |
| 배포 적합성 | **★ 높음** | 낮음(학습 자원 필요) |

**AWQ는 PTQ의 한 종류**입니다. 이미 학습된 모델을 *소수의 예시(calibration)* 로 한 번 훑어 4bit로 바꿉니다 — 그래서 배포에 잘 맞습니다.

배포를 줄이자!

## 2. AWQ 원리 — "모든 가중치가 똑같이 중요하진 않다"

AWQ(**A**ctivation-a**W**are **Q**uantization)의 핵심 아이디어:
> 선형층 출력은 `y = W·x`입니다. 이때 **`x` 값이 큰 채널**일수록, 그 채널 가중치를 양자화할 때 생기는 오차가 출력에 더 크게 반영됩니다.  
> 즉, 오차의 대부분은 이런 **중요(salient) 채널**에서 발생합니다. 
> 여기서 **채널(channel)**은 입력 벡터 `x`의 각 원소(인덱스 `j`)를 의미한다. 선형층은 `y = Σⱼ W[:,j]·x[j]`로 계산되므로, `|x[j]|`가 큰 채널일수록 해당 채널의 가중치 양자화 오차가 출력에 더 크게 증폭된다.

즉 **활성값이 큰 소수 채널의 가중치를 보존**하면, 같은 4bit라도 오차가 크게 줍니다. 아래 셀로 직접 확인합니다.

x - activation 함수

오차 MA(Mean Absolute)

중요한 weight를 잘 보호해서 오차줄이자!!!

In [ ]:
# ── AWQ 직관 데모: salient 채널 보존이 출력 오차를 줄인다 ─────
import numpy as np
np.random.seed(0)

# 데모에서 사용할 선형층 크기 설정
# - D_in : 입력 채널 수(입력 벡터 x의 길이)
# - D_out: 출력 채널 수(출력 벡터 y의 길이)
# 이미 같은 이름의 변수가 노트북에 있더라도, 이 셀 실행 시 재현성을 위해 명시적으로 다시 지정합니다.
D_in, D_out = 256, 64

# 가중치 행렬 W 생성 (shape: [D_out, D_in])
# - np.random.randn: 평균 0, 표준편차 1인 정규분포 샘플
# - * 0.1           : 가중치 스케일을 줄여(표준편차 0.1) 과도한 값 방지
# - astype(float32) : 메모리 절약 + 딥러닝에서 흔히 쓰는 dtype으로 통일
W = (np.random.randn(D_out, D_in) * 0.1).astype(np.float32)

# 입력 활성값 벡터 x 생성 (shape: [D_in])
# - 각 입력 채널의 활성값을 정규분포에서 샘플링
# - 이후 코드에서 일부 채널(salient)을 크게 만들어 AWQ 직관을 확인
x = np.random.randn(D_in).astype(np.float32)

# ─────────────────────────────────────────────────────────────
# salient 채널(중요 채널) 선택 단계
# AWQ 직관: |x_j|가 큰 입력 채널 j는 출력 y = W·x에 더 큰 영향(오차 증폭)을 준다.
# 따라서 상위 소수 채널만 보호해도 전체 출력 오차를 크게 줄일 수 있다.
# ─────────────────────────────────────────────────────────────

# 1) 전체 입력 채널(D_in) 중 "몇 개를 salient로 볼지" 결정
#    - 여기서는 데모 규칙으로 상위 1%를 선택
#    - D_in이 아주 작을 때 0개가 되지 않도록 최소 1개 보장
n_salient = max(1, D_in // 100)

# 2) 채널 중요도 계산: |x| (활성값 절댓값)
#    - 활성값의 부호(+/-)보다 "크기"가 영향력에 중요하므로 절댓값 사용
#    - np.argsort(-np.abs(x)): 절댓값이 큰 순서대로 인덱스 정렬
#    - [:n_salient]: 상위 n_salient개 채널 인덱스만 추출
salient = np.argsort(-np.abs(x))[:n_salient]

# 3) 선택된 salient 채널의 활성값을 인위적으로 키움(×10)
#    - 데모 목적: "중요 채널이 매우 큰 활성" 상황을 명확히 재현
#    - 이후 양자화 오차 비교에서,
#      salient를 보존하지 않으면 오차가 커지고, 보존하면 오차가 줄어드는 효과가 더 뚜렷해진다.
x[salient] *= 10.0

def quant4(w):
    """
    간단화한 대칭 4bit 양자화(-7..7).
    입력: 실수 가중치 텐서/배열 w
    출력: 4bit 격자에 맞춰 반올림된 가중치
    """
    qmax = 7  # 대칭 4bit에서 사용하는 최대 정수 레벨 ([-7, 7], 총 15개 유효 레벨)
    
    # 전체 텐서의 절댓값 최댓값을 기준으로 스케일 계산 (per-tensor scale)
    # +1e-8은 w가 모두 0일 때 0으로 나누는 문제를 방지
    scale = np.abs(w).max() / qmax + 1e-8
    
    # 1) 스케일로 정규화
    # 2) 반올림하여 정수 격자에 투영
    # 3) [-7, 7] 범위로 클리핑
    # 4) 다시 scale을 곱해 원래 스케일의 양자화 값으로 복원
    return np.round(w / scale).clip(-qmax, qmax) * scale

# 기준 출력: 양자화하지 않은 원본 가중치(fp32)로 계산한 정답 출력
y_ref = W @ x

# (A) 전부 4bit 양자화
# - W 전체를 quant4로 양자화한 뒤 출력 계산
# - y_ref와의 평균 절대 오차(MAE)로 성능 저하 측정
err_naive = np.abs(quant4(W) @ x - y_ref).mean()

# (B) salient 채널 보존 + 나머지 4bit
# - 먼저 전체를 4bit로 양자화
# - 이후 중요한 채널(salient)에 해당하는 "열(column)"만 원본 값으로 복원
W_prot = quant4(W).copy()
W_prot[:, salient] = W[:, salient]

# - 보호 전략 적용 후 출력 오차(MAE) 측정
err_prot = np.abs(W_prot @ x - y_ref).mean()

# 결과 요약 출력
print(f'salient 채널: {n_salient}/{D_in} ({100*n_salient/D_in:.0f}%)')
print(f'  (A) 전부 4bit   : {err_naive:.5f}')
print(f'  (B) salient 보존 : {err_prot:.5f}')

# 오차 감소율(%): salient 보존 전략이 naive 대비 얼마나 개선되는지
print(f'  → 오차 감소율    : {(1 - err_prot/err_naive)*100:.1f}%')

salient 채널: 2/256 (1%)
  (A) 전부 4bit   : 0.56436
  (B) salient 보존 : 0.18036
  → 오차 감소율    : 68.0%


### 2-1. 실행 결과 읽기 — 숫자가 말하는 것

방금 출력의 네 줄을 그대로 해부하면:

| 출력 | 의미 |
|---|---|
| `salient 채널: 2/256 (1%)` | 입력 256채널 중 **활성값 절댓값이 가장 큰 2개**를 salient로 지정(코드에서 그 채널의 `x`를 ×10으로 키움). 실전에선 이걸 **calibration**으로 측정합니다. |
| `(A) 전부 4bit : 0.56436` | 가중치 `W`를 **전부 단순 4bit**로 양자화했을 때의 출력 평균오차 `mean(abs(ŷ−y))`. |
| `(B) salient 보존 : 0.18036` | salient **2채널의 가중치 열만 원본(fp32)** 으로 두고 나머지는 4bit일 때의 오차. **약 1/3로 감소.** |
| `→ 오차 감소율 : 68.0%` | `(1 − 0.18036/0.56436)×100`. **단 1% 채널 보존으로 출력 오차의 2/3가 사라짐.** |

**왜 이렇게 극적인가?** 출력은 `y = Σⱼ W[:,j]·x[j]` 입니다. 채널 `j`의 양자화 오차가 출력에 미치는 영향은 `ΔW[:,j] · x[j]` — 즉 **그 채널의 활성값 `x[j]`에 비례**합니다. 활성값이 10배 큰 salient 채널은 같은 가중치 오차라도 출력 오차를 10배로 증폭하므로, **전체 오차를 소수의 salient 채널이 지배**합니다. 그 채널만 정밀하게 지키면 지배항이 사라져 오차가 급감합니다.

> ⚠️ **숫자 자체는 예시값입니다.** `np.random.seed(0)` + 특정 `W·x` + ×10 배율에서 나온 값이라 절대 수치(0.56/0.18/68%)는 설정에 따라 달라집니다. 하지만 **"소수 salient 채널 보존 → 큰 오차 감소"** 라는 방향성은 견고합니다 — 이것이 AWQ의 출발점입니다.

**위 데모의 교훈**: 단 1% 채널을 보존했을 뿐인데 출력 오차가 크게 줄었습니다.

단, 실제 AWQ는 위처럼 *일부만 fp16으로 섞지(mixed precision)* 않습니다 — 하드웨어에 비효율적이기 때문입니다. 대신 **스케일링**을 씁니다: salient 채널의 가중치를 `s`배 키워 양자화 정밀도를 더 주고, 대응 입력을 `1/s`배로 줄여 **출력은 보존**합니다. 결과적으로 **전부 4bit를 유지**하면서 중요한 채널만 더 정밀해집니다. `s`는 calibration 데이터로 채널별 활성값 크기를 보고 정합니다.

### 2-2. 스무딩(smoothing) — AWQ가 실제로 하는 일

위 데모는 "중요한 채널을 fp16으로 **보존**하면 오차가 준다"는 **직관**을 보였습니다. 하지만 일부만 fp16으로 섞는 방식(mixed precision)은 하드웨어에 비효율적입니다. 그래서 실제 AWQ는 **전부 4bit를 유지**하면서 같은 효과를 내는 **스무딩(smoothing)** 을 씁니다.

**한 줄 정의** — 채널마다 **스케일 `s`를 정해** 활성값은 줄이고(`x ÷ s`) 대응 가중치는 키워(`W × s`) 출력은 유지하면서, 가중치 분포를 고르게 펴서 4bit 양자화 오차를 줄이는 것.

**무엇을 하나.** 선형층은 `y = W·x` 입니다. 채널마다 스케일 `s`를 정해서 **활성은 나누고**(`x ÷ s`), 대응하는 **가중치는 곱합니다**(`W × s`). 곱이 그대로라 **출력은 변하지 않습니다**:

$$ (W \times s)\cdot(x \div s) = W\cdot x $$

수학적으로 동일한 다른 표현으로 바꾸는 것뿐입니다.

**왜 하나.** 4bit는 표현 격자가 16칸뿐이라 **너무 작은 가중치는 0으로 뭉개집니다.** 그 작은 가중치가 큰 활성과 곱해지는 **salient 채널**이면, 0이 되는 순간 출력이 크게 틀어집니다. 스무딩은 그 가중치를 `s`배 **키워** 격자에서 **살아남게** 하고, 활성을 같은 비율로 줄여 출력을 보존합니다.

**이름의 뜻.** 특정 채널만 유난히 작거나 큰 가중치 분포의 **들쭉날쭉함을 고르게(smooth) 펴서** 양자화하기 좋은 분포로 만들기 때문에 '스무딩'이라 부릅니다.

아래 셀에서 4채널 예시로 직접 확인합니다(작은 가중치 0.05가 양자화에서 0으로 뭉개졌다가, 스무딩으로 살아나는 장면).

In [ ]:
# ── 스무딩 데모: 출력은 그대로, 중요 채널을 살려 4bit 오차를 줄인다 ──
import numpy as np

def quant4(w):
    """대칭 4bit(레벨 -7..7) 양자화 — 위의 데모와 동일 방식"""
    # 최댓값 기준 스케일 계산 (per-tensor)
    scale = np.abs(w).max() / 7
    # 스케일로 정규화 → 반올림 → [-7,7] 클리핑 → 스케일 복원
    return np.round(w / scale).clip(-7, 7) * scale

# 4채널 선형층(출력 1개). 채널4가 salient(활성값 30). 가중치(0.05)는 작지만 활성이 커서 기여가 큼.
# 입력: [1.0, 1.0, 1.0, 30.0] — 4번째 채널의 활성값이 매우 큼
x = np.array([1.0, 1.0, 1.0, 30.0])
# 가중치: [0.9, 0.9, 0.9, 0.05] — 4번째 채널의 가중치는 매우 작음
w = np.array([0.9, 0.9, 0.9, 0.05])
# 정답 출력 계산 (w·x의 내적)
y_true = w @ x
print(f'정답 출력 y = {y_true:.2f}  (채널4 기여 = 0.05x30 = 1.5)')

# (A) 스무딩 없이 그냥 4bit 양자화
wq = quant4(w)  # 전체 가중치를 4bit로 양자화
yA = wq @ x     # 양자화된 가중치로 출력 계산
print(f'\n[A] 스무딩 없음  : 양자화 가중치 {np.round(wq,3).tolist()}')
print(f'    -> 채널4 가중치 0.05가 0으로 뭉개짐 -> y={yA:.2f}, 오차 {abs(yA-y_true):.2f} ({abs(yA-y_true)/y_true*100:.0f}%)')

# (B) AWQ 스무딩: 채널4 가중치 x8, 활성 ÷8 (곱은 그대로 -> 출력 불변)
# 채널별 스케일 벡터 (채널4만 8배)
s = np.array([1, 1, 1, 8.0])
# 가중치는 스케일로 키우고(×s), 활성은 스케일로 줄임(÷s)
# 수학적으로: (w·s)·(x÷s) = w·x (출력은 동일)
w2, x2 = w * s, x / s
wq2 = quant4(w2)  # 스무딩 후 가중치를 4bit로 양자화
yB = wq2 @ x2     # 양자화된 가중치로 출력 계산
print(f'\n[B] AWQ 스무딩 후: 채널4 가중치 0.05->{w2[3]:.1f}, 활성 30->{x2[3]:.2f}')
print(f'    -> 양자화 가중치 {np.round(wq2,3).tolist()} (채널4 살아남음)')
print(f'    -> y={yB:.2f}, 오차 {abs(yB-y_true):.2f} ({abs(yB-y_true)/y_true*100:.1f}%)')
print(f'\n오차: {abs(yA-y_true):.2f} -> {abs(yB-y_true):.2f}  (스무딩이 중요 채널을 지켜 오차 급감)')

정답 출력 y = 4.20  (채널4 기여 = 0.05x30 = 1.5)

[A] 스무딩 없음  : 양자화 가중치 [0.9, 0.9, 0.9, 0.0]
    -> 채널4 가중치 0.05가 0으로 뭉개짐 -> y=2.70, 오차 1.50 (36%)

[B] AWQ 스무딩 후: 채널4 가중치 0.05->0.4, 활성 30->3.75
    -> 양자화 가중치 [0.9, 0.9, 0.9, 0.386] (채널4 살아남음)
    -> y=4.15, 오차 0.05 (1.3%)

오차: 1.50 -> 0.05  (스무딩이 중요 채널을 지켜 오차 급감)


### 2-3. 실행 결과 읽기 — 스무딩 전후

| | 채널4 가중치 | 4bit 양자화 후 | 출력 오차 |
|---|---|---|---|
| **스무딩 없음** | 0.05 | **0 (뭉개짐)** | 1.50 (36%) |
| **스무딩 후(×8)** | 0.4 | 0.386 (살아남음) | 0.05 (1.3%) |

작은 0.05를 0.4로 키워두니 4bit로 반올림해도 **살아남고**, 활성을 ÷8로 줄여 **출력(4.2)은 유지**됩니다. 그 결과 오차가 36% → 1.3%로 급감했습니다. ([6]은 'fp16으로 보존', 여기[2-2]는 '전부 4bit + 스케일' — **결과는 같고 방식이 하드웨어 친화적**입니다.)

실제 양자화 때 이 과정이 레이어마다 보입니다.
- **Calibrating** — 어느 채널이 중요한지(활성이 큰지) 측정 → **채널별 `s` 결정**.
- **Smoothing** — 그 `s`로 `W×s`, `x÷s` 재배치 수행(위 [B]).
- **Propagating** — 바뀐 값을 다음 레이어 입력으로 전파.

> ⚠️ 숫자(×8, 0.4 등)는 설명을 위한 예시값입니다. 실제 `s`는 calibration이 채널별 활성 크기를 보고 정합니다. 방향성 — **출력을 바꾸지 않고 중요 가중치를 양자화하기 쉬운 크기로 '고르게 펴는' 것** — 이 스무딩의 핵심입니다.

## 3. AWQ · GPTQ · compressed-tensors

| 이름 | 무엇 | 비고 |
|---|---|---|
| **AWQ** | activation-aware 4bit **방법** | 본 과정의 메인 |
| **GPTQ** | 2차 정보(Hessian approximation) 기반 4bit **방법** | AWQ의 대안. 둘 다 PTQ 4bit |
| **compressed-tensors** | 양자화 결과를 담는 **저장 포맷** | AWQ/GPTQ 결과를 이 포맷으로 저장 → vLLM이 바로 읽음 |

> 정리: **AWQ·GPTQ는 "방법"**, **compressed-tensors는 "저장 컨테이너"**. 우리는 *AWQ 방법*으로 만들어 *compressed-tensors 포맷*으로 저장하고 *vLLM*으로 서빙합니다.

## 4. VLM 양자화의 특수성 — 무엇을 양자화하고 무엇을 남기나

VLM은 크게 세 부분입니다.

```
  [ 비전 타워 ]  →  [ 프로젝터 ]  →  [ LLM (텍스트) ]
   이미지 인코딩      차원 정합        대부분의 파라미터·연산
```

- **LLM 선형층**: 파라미터·연산의 대부분 → **여기를 4bit로 양자화**(이득이 큼).
- **비전 타워 / 프로젝터**: 상대적으로 작고, **지각 품질에 민감** → **양자화에서 제외(`ignore`)**.

> 그래서 공개 Qwen3-VL AWQ 체크포인트들도 동일하게 **비전 모듈을 ignore**합니다. Day2-3에서 `llm-compressor`의 `ignore` 목록에 비전 모듈을 넣는 이유입니다. *작은 모듈을 무리하게 4bit로 깎아 이미지 이해를 망치는 것*을 피하는 실용적 선택입니다.

## 5. Calibration — 왜 소수 샘플로 충분한가

AWQ는 **“어떤 채널이 중요한지”**만 파악하면 됩니다.  
그래서 모델에 **대표 샘플 몇 백 개**(예: ChartQA 이미지+질문 128~512개)를 넣어 보고, 채널별 활성값 크기를 기록합니다. 이 과정을 **calibration(캘리브레이션)**이라고 합니다.

- **학습이 아닙니다.** (가중치 업데이트/역전파 없음)
- **순전파만** 하므로 비교적 **빠릅니다**. (보통 수 분~수십 분)
- 샘플은 실제 서비스 데이터와 **비슷한 분포**여야 효과적입니다.  
    → 그래서 우리 실습에서는 **ChartQA로 calibration**합니다.

## 6. 우리가 만들 결과물 미리보기 — compressed-tensors 설정

Day2-3에서 만들 AWQ 모델은 `config.json` 안에 아래 같은 **양자화 설정**을 담습니다. vLLM은 이 설정을 보고 4bit 가중치를 어떻게 풀지 압니다.

In [ ]:
# ── AWQ(compressed-tensors) 양자화 설정 예시 ──
# 실제 Day2-3에서 llm-compressor가 이런 구조를 자동 생성합니다.

# compressed-tensors 형식의 AWQ 설정을 미리보기용으로 정의
quantization_config_preview = {
    'quant_method': 'compressed-tensors',   # 저장/서빙 포맷 이름
    'format': 'pack-quantized',             # 4bit 값을 패킹해서 저장
    'config_groups': {
        'group_0': {
            'targets': ['Linear'],          # 양자화할 모듈 대상: 선형층
            'weights': {
                'num_bits': 4,              # 가중치를 4bit로 양자화
                'type': 'int',              # 정수 양자화
                'symmetric': True,          # 대칭 양자화 사용
                'strategy': 'group',        # 그룹 단위 스케일 적용
                'group_size': 128,          # 한 그룹에 포함될 가중치 개수
            },
        }
    },
    # 비전 타워와 출력 헤드는 양자화에서 제외
    'ignore': ['lm_head', 're:.*visual.*'],
}

import json

# 보기 좋게 JSON 형태로 출력
print(json.dumps(quantization_config_preview, indent=2, ensure_ascii=False))

{
  "quant_method": "compressed-tensors",
  "format": "pack-quantized",
  "config_groups": {
    "group_0": {
      "targets": [
        "Linear"
      ],
      "weights": {
        "num_bits": 4,
        "type": "int",
        "symmetric": true,
        "strategy": "group",
        "group_size": 128
      }
    }
  },
  "ignore": [
    "lm_head",
    "re:.*visual.*"
  ]
}


### 6-1. 실행 결과 읽기 — 이 설정을 vLLM이 어떻게 읽나

이 JSON은 AWQ 모델 `config.json` 안의 `quantization_config`가 될 구조입니다. 필드별로:

| 필드 | 값 | 뜻 |
|---|---|---|
| `quant_method` | `compressed-tensors` | 저장 포맷 이름. **vLLM이 이 키를 보고** 압축 가중치 해제 방식을 자동 선택(별도 `--quantization` 플래그 불필요). |
| `format` | `pack-quantized` | 4bit 정수 8개를 **int32 한 칸에 패킹**해 저장 → 로드 시 언팩. 디스크·VRAM 절약의 실체. |
| `config_groups.group_0.targets` | `['Linear']` | **`nn.Linear` 모듈만** 양자화 대상. |
| `weights.num_bits` / `type` | `4` / `int` | **4비트 정수** 양자화(부동소수 아님). |
| `weights.symmetric` | `true` | **대칭** 양자화(zero-point=0, 스케일만 사용) → 스킴 이름 **`W4A16`**. 비대칭이면 `W4A16_ASYM`. |
| `weights.strategy` / `group_size` | `group` / `128` | 채널 전체가 아니라 **가중치 128개 묶음(group)마다 별도 스케일**. 정밀도를 크게 올리는 핵심 — 4bit 정확도의 대부분이 여기서 나옵니다(메타데이터는 소폭 증가). |
| `ignore` | `['lm_head', 're:.*visual.*']` | **출력 head와 비전 타워는 양자화 제외**(4번 절의 VLM 양자화 범위). `re:.*visual.*` 는 이름에 `visual`이 든 모듈 전부를 정규식으로 매칭. |

**"A16"은 어디에?** 이 설정엔 **가중치(W4)만** 적혀 있습니다 — 활성값은 16bit(fp16/bf16)로 그대로 둡니다. 그래서 스킴이 **W4A16**(weight 4bit, activation 16bit)이고, 추론 시 4bit 가중치를 16bit로 풀어 곱합니다.

> 📌 이건 **개념 미리보기**라 ignore가 `re:.*visual.*` 한 줄이지만, **Day2-3 실제 산출물**에선 공개 Qwen3-VL 체크포인트 관례에 맞춰 `["lm_head", "re:visual.*", "re:model.visual.*"]` 로 **비전 경로를 더 정밀하게** 지정합니다. 구조(4bit·group128·symmetric·Linear 대상)는 동일합니다.

## 7. 정리 + 다음 교시 예고

- **PTQ**(학습 후 양자화)는 배포에 맞고, **AWQ**는 그 대표 방법입니다.
- AWQ는 **활성값이 큰 채널을 스케일링으로 보호**해 4bit를 유지하면서 오차를 줄입니다.
- VLM에선 **LLM 선형층만 4bit**, **비전 타워·프로젝터는 ignore**.
- **calibration**은 소수 샘플로 활성값 통계만 모으는 빠른 과정입니다.

### 다음 교시 — Day2-2 · 환경·모델 반입
RunPod A100에 **uv + Python 3.12 + venv** 환경을 만들고 `llm-compressor`를 설치합니다. Day1에서 Hub에 올린 **병합 4B를 반입**하고 베이스 **8B를 다운로드**한 뒤, **calibration set**을 준비합니다.

> ✅ **체크포인트**: ① PTQ와 QAT를 구분한다 ② salient 채널 보존이 왜 오차를 줄이는지 설명할 수 있다 ③ VLM에서 무엇을 ignore하는지 안다 ④ calibration이 학습이 아님을 안다.